# Part 2 — real model inference on all seven shelf photographs

This notebook calls both models rather than replaying saved detections. **It has not been executed yet.** Run it, retain the executed notebook, and return its downloaded results ZIP.

1. In Colab choose **Runtime → Change runtime type → T4 GPU** (subject to free availability).
2. Run the cells in order. A Roboflow API key is requested privately using `getpass`; do not paste it into chat, a code cell, or GitHub. Use an account with available inference allowance; do not purchase credits for this assignment.
3. The LLM is the downloadable **Qwen2.5-VL-3B-Instruct**, running on the Colab GPU without a paid LLM API. Initial model download is several GB.
4. Final output includes raw model responses, prompt, normalized detections, counts, code, and a self-contained overlay webpage. **Counts may differ from our interactive ChatGPT/browser results.** Keep the two experiments separately labelled.

Sources: [Qwen model card](https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct), [Roboflow model by twar](https://universe.roboflow.com/twar/empty-shelf-detection-r8cfz), CC BY 4.0 for the Roboflow project. No accuracy claim is established without reference annotations.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.1', 'accelerate>=0.34,<2', 'qwen-vl-utils==0.0.8',
    'inference-sdk>=0.64,<1', 'Pillow', 'requests'])
print('Dependencies installed. If Colab requests a runtime restart, restart and continue with the next cell.')


In [ ]:
import io, json, zipfile, shutil, hashlib, datetime, importlib.metadata, getpass
from pathlib import Path
import requests
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Choose Runtime → Change runtime type → T4 GPU, then run again. Do not buy compute units.')
ROOT=Path('/content/shelf_inference_assignment')
ROOT.mkdir(exist_ok=True)
url='https://codeload.github.com/MVEST22/-ai-accounting-ml-assignment./zip/refs/heads/main'
r=requests.get(url,timeout=120);r.raise_for_status()
with zipfile.ZipFile(io.BytesIO(r.content)) as archive:
    for info in archive.infolist():
        if '..' in Path(info.filename).parts or Path(info.filename).is_absolute():
            raise ValueError('Unexpected archive path')
    archive.extractall(ROOT)
BASE=next(ROOT.glob('*/Part 2 - shelves'))
IMAGES=sorted(BASE.glob('zone*/*.jpg'))
assert len(IMAGES)==7
OUT=ROOT/'programmatic_results'
OUT.mkdir(exist_ok=True)
(OUT/'raw_llm').mkdir(exist_ok=True)
(OUT/'raw_roboflow').mkdir(exist_ok=True)
print('GPU:',torch.cuda.get_device_name(0))
print('Loaded',len(IMAGES),'photos from the repository.')


In [ ]:
from inference_sdk import InferenceHTTPClient, InferenceConfiguration
KEY=getpass.getpass('Roboflow API key (private input; requires available inference allowance): ')
if not KEY.strip():
    raise RuntimeError('No key entered. Obtain it from your own Roboflow workspace; do not share it in chat.')
client=InferenceHTTPClient(api_url='https://serverless.roboflow.com',api_key=KEY)
client.configure(InferenceConfiguration(confidence_threshold=0.25,iou_threshold=0.5,api_key_transport='header'))
cv={'source':'Programmatic Roboflow SDK inference executed in Colab',
    'model_id':'empty-shelf-detection-r8cfz/1','confidence_threshold':0.25,'overlap_threshold':0.5,'images':[]}
for path in IMAGES:
    try:
        raw=client.infer(str(path),model_id=cv['model_id'])
    except Exception:
        raise RuntimeError('Roboflow inference failed. Check your key, model access and available allowance. No zero-count result was substituted.') from None
    (OUT/'raw_roboflow'/(path.stem+'.json')).write_text(json.dumps(raw,indent=2))
    predictions=raw['predictions']
    retained=[b for b in predictions if b['class']=='Empty-space' and float(b['confidence'])>=0.25]
    cv['images'].append({'image':path.relative_to(BASE).as_posix(),'width':640,'height':640,'predictions':retained})
    print(path.name[:23],len(retained),'empty areas')
(OUT/'roboflow_programmatic.json').write_text(json.dumps(cv,indent=2))
KEY=None
client=None
print('Specialized-model outputs saved. The key is not saved in any output file.')


In [ ]:
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
MODEL='Qwen/Qwen2.5-VL-3B-Instruct'
torch.manual_seed(0)
model=Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL,torch_dtype=torch.float16,device_map='auto',attn_implementation='sdpa')
model.eval()
processor=AutoProcessor.from_pretrained(MODEL,min_pixels=256*28*28,max_pixels=768*28*28)
PROMPT='Identify contiguous visibly empty front-facing shelf areas needing replenishment in this photograph. Count each area once, not individual missing products. Exclude the floor, black image borders, normal shelf edges and ordinary narrow spacing between stocked products. Stock pushed to the back with exposed shelf front may be a replenishment area. Return only a JSON object with a boxes array. For each area return x1, y1, x2, y2 and reason. Coordinates MUST be normalized from 0 to 1000 across the full image: top-left (0,0), bottom-right (1000,1000). Each box must have x1<x2 and y1<y2. If there are no empty areas return {"boxes": []}. Do not copy or refer to any other model results.'
(OUT/'llm_prompt.txt').write_text(PROMPT)
llm={'source':'Programmatic Qwen visual inference executed in Colab; no other model outputs included in prompts',
     'model_id':MODEL,'unit':'contiguous empty front-facing shelf areas, not missing product units','images':[]}
for path in IMAGES:
    messages=[{'role':'user','content':[{'type':'image','image':Image.open(path).convert('RGB')},{'type':'text','text':PROMPT}]}]
    text=processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    image_inputs,video_inputs=process_vision_info(messages)
    inputs=processor(text=[text],images=image_inputs,videos=video_inputs,padding=True,return_tensors='pt').to('cuda')
    with torch.inference_mode():
        generated=model.generate(**inputs,max_new_tokens=2048,do_sample=False)
    trimmed=[out[len(inp):] for inp,out in zip(inputs.input_ids,generated)]
    response=processor.batch_decode(trimmed,skip_special_tokens=True,clean_up_tokenization_spaces=False)[0]
    (OUT/'raw_llm'/(path.stem+'.txt')).write_text(response)
    start,end=response.find('{'),response.rfind('}')
    try:
        parsed=json.loads(response[start:end+1])
        boxes=[]
        for b in parsed['boxes']:
            x1,y1,x2,y2=[float(b[k]) for k in ['x1','y1','x2','y2']]
            if not (0<=x1<x2<=1000 and 0<=y1<y2<=1000):
                raise ValueError('Invalid normalized coordinates')
            boxes.append({'x1':x1*.64,'y1':y1*.64,'x2':x2*.64,'y2':y2*.64,'reason':str(b.get('reason',''))})
    except (ValueError,KeyError,TypeError):
        raise RuntimeError('LLM response was not valid JSON/boxes for '+path.name+'. Raw response was saved. Do not invent a zero count; return the raw response for troubleshooting.') from None
    llm['images'].append({'image':path.relative_to(BASE).as_posix(),'width':640,'height':640,'boxes':boxes})
    (OUT/'llm_programmatic.json').write_text(json.dumps(llm,indent=2))
    print(path.name[:23],len(boxes),'empty areas')
    del inputs,generated,trimmed
    torch.cuda.empty_cache()
print('Both models have now run on all seven photographs.')


In [ ]:
import csv, base64
assert len(llm['images'])==len(cv['images'])==7
lmap={i['image']:i for i in llm['images']};cmap={i['image']:i for i in cv['images']}
data=[]
for path in IMAGES:
    name=path.relative_to(BASE).as_posix()
    right=[]
    for b in cmap[name]['predictions']:
        x1=b['x']-b['width']/2;y1=b['y']-b['height']/2
        x2=x1+b['width'];y2=y1+b['height']
        assert 0<=x1<x2<=640 and 0<=y1<y2<=640
        right.append({'x1':x1,'y1':y1,'x2':x2,'y2':y2,'confidence':b['confidence']})
    data.append({'image':name,'image_data':'data:image/jpeg;base64,'+base64.b64encode(path.read_bytes()).decode(),
                 'zone':path.parent.name,'name':path.name.split('WA')[1][:4],'llm':lmap[name]['boxes'],'cv':right})
with (OUT/'comparison_programmatic.csv').open('w',newline='') as f:
    w=csv.writer(f);w.writerow(['zone','filename','qwen_llm_empty_areas','roboflow_empty_areas'])
    for i in data:w.writerow([i['zone'],Path(i['image']).name,len(i['llm']),len(i['cv'])])
html=(BASE/'comparison_template.html').read_text()
html=html.replace('Interactive ChatGPT visual estimates','Qwen2.5-VL-3B · actual programmatic run').replace('LLM · ChatGPT','LLM · Qwen2.5-VL')
html=html.replace('Browser outputs supplied by the user; no hosted API call was used to produce these saved results. LLM boxes were created interactively after the specialized outputs were available, so this is not a blinded accuracy experiment.',
 'Both models executed programmatically in Colab. Qwen prompts contained no specialized detections. No ground truth was supplied; this is not a validated accuracy measurement.')
# The recommendation must be reassessed against the newly generated detections.
start=html.index('<article><h2>Recommendation for store employees')
end=html.index('</article>',start)+len('</article>')
html=html[:start]+'<article><h2>Recommendation pending visual review</h2><p>Compare both overlays against each original photo before selecting results for employees. Counts are empty areas, not exact missing-unit quantities. Confirm alerts manually and do not assume the higher count is more accurate.</p></article>'+html[end:]
# Remove comments specific to the earlier interactive detections.
html=html.replace("if(i.name==='0009')article.append(element('p','Same count, different locations: compare the LLM’s central milk-carton gap with the specialized box near the bottom right.'));",'')
html=html.replace("if(i.name==='0011')article.append(element('p','Inspect the LLM’s gap on the second shelf against the specialized box lower down. Neither is a verified reference annotation.'));",'')
html=html.replace('/*DATA*/[]',json.dumps(data).replace('<','\\u003c'))
(OUT/'comparison_programmatic.html').write_text(html)
manifest={'executed_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'llm_model':MODEL,'roboflow_model':cv['model_id'],'confidence_threshold':.25,'overlap_threshold':.5,
 'packages':{n:importlib.metadata.version(n) for n in ['torch','transformers','qwen-vl-utils','inference-sdk']},
 'images_sha256':{p.relative_to(BASE).as_posix():hashlib.sha256(p.read_bytes()).hexdigest() for p in IMAGES}}
(OUT/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
shutil.copyfile(BASE/'Part_2_Programmatic_Inference.ipynb',OUT/'inference_source_unexecuted_template.ipynb')
(OUT/'README.txt').write_text('Actual programmatic results. Open comparison_programmatic.html locally and review all boxes. Also download your executed Colab notebook using File > Download > Download .ipynb. The included template notebook is source only and is not execution evidence. Send results ZIP and executed notebook back for final review and recommendation. No API key is included.')
archive=shutil.make_archive(str(ROOT/'part2_programmatic_results'),'zip',OUT)
print('LLM counts:',[len(i['llm']) for i in data])
print('Roboflow counts:',[len(i['cv']) for i in data])
from google.colab import files
files.download(archive)
